In [1]:
import os
os.chdir("../")
%pwd

'd:\\Deep_Learning_Object_Detection\\MLOPs\\pneumonia-segmentation'

In [2]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir:       Path
    data_dirs:      tuple
    out_train_dir:  Path
    out_valid_dir:  Path
    out_infer_dir:  Path
    params_image_size: int
    params_skip_background_ratio: float
    params_slice_interval: int
    params_valid_size: float
    params_infer_size: float

In [3]:
import os
from pathlib import Path
from dotenv import load_dotenv
load_dotenv()

from pneumonia_segmentation.constants import *
from pneumonia_segmentation.utils.common import read_yaml, create_directories

class ConfigurationManger:
    def __init__(self, 
                 config_filepath = CONFIG_FILE_PATH,
                 params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])
    
    def _parse_data_sources(self):
        raw = os.getenv("DATA_SOURCES", "")
        
        sources = []
        for entry in raw.split(","):
            name, source_type, ingestion_type, source = entry.strip().split(":")
            sources.append({
                "name": name,
                "source_type": source_type,
                "ingestion_type": ingestion_type,
                "source": source
            })
        return sources

    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation_config
        params = self.params.data_transformation_params
        create_directories([config.root_dir])
        
        sources   = self._parse_data_sources()
        data_dirs = [
            {
                "path": Path(self.config.data_ingestion_config.root_dir) / s["name"],
                "source_type": s["source_type"]
            }
            for s in sources
        ]
        
        data_transformation_config = DataTransformationConfig(
            root_dir               = config.root_dir,
            data_dirs              = data_dirs,
            out_train_dir          = config.out_train_dir,
            out_valid_dir          = config.out_valid_dir,
            out_infer_dir          = config.out_infer_dir,
            params_image_size      = params.image_size,
            params_skip_background_ratio = params.skip_background_ratio,
            params_slice_interval  = params.slice_interval,
            params_valid_size      = params.valid_size,
            params_infer_size      = params.infer_size
        )

        return data_transformation_config

In [ ]:
import os, sys
from tqdm.auto import tqdm
from pneumonia_segmentation import logging
from pneumonia_segmentation.exception import CustomException

from pneumonia_segmentation.utils.covid_ct_processing import *
from pneumonia_segmentation.adapters.factories import TransformationAdapterFactory

class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config   = config
        self.adapters = [
            TransformationAdapterFactory.get_adapter(d["source_type"], d["path"])
            for d in self.config.data_dirs
        ]
    
    def _prepare_output_dirs(self):
        for d in [self.config.out_train_dir, self.config.out_valid_dir, self.config.out_infer_dir]:
            os.makedirs(os.path.join(d, "img"), exist_ok=True)
            os.makedirs(os.path.join(d, "msk"), exist_ok=True)
            
    def transform(self):
        try:
            self._prepare_output_dirs()
            img_counter = 1
            
            for adapter in self.adapters:
                for ct_vol, lung_vol, infect_vol in adapter.get_data_generator():
                    for i in range(0, ct_vol.shape[2], self.config.params_slice_interval):
                        ct_slice     = ct_vol[:, :, i]
                        lung_slice   = lung_vol[:, :, i] if lung_vol is not None else None
                        infect_slice = infect_vol[:, :, i]
                        
                        if should_skip_slice(infect_slice, self.config.params_skip_background_ratio):
                            continue
                            
                        mask_bin = (infect_slice > 0).astype("uint8")
                        
                        if lung_slice is not None:
                            ct_crop, mask_crop = crop_to_lung_roi(ct_slice, mask_bin, lung_slice)
                        else:
                            ct_crop, mask_crop = resize_full_slice(ct_slice, mask_bin, self.config.params_image_size)
                
                        img, mask = normalize_and_colormap(ct_crop, mask_crop, self.config.params_image_size)
                        
                        self._save_pair(img, mask, self._get_output_dir(), img_counter)
                        img_counter += 1
                
                if img_counter > 1:
                    logging.info(f"Data transformation completed: {img_counter - 1} pairs saved")  
        except Exception as e:
            raise CustomException(e, sys)
    
    def _get_output_dir(self) -> str:
        r = random.random()
        if r < self.config.params_infer_size:
            return self.config.out_infer_dir
        elif r < self.config.params_infer_size + self.config.params_valid_size:
            return self.config.out_valid_dir

        return self.config.out_train_dir

    def _save_pair(self, img, mask, out_dir: str, counter: int) -> None:
        img_saved = cv2.imwrite(os.path.join(out_dir, "img", f"scan_{counter:05d}.png"), img)
        mask_saved= cv2.imwrite(os.path.join(out_dir, "msk", f"mask_{counter:05d}.png"), mask)
        
        if not img_saved or not mask_saved:
            logging.warning(f"Failed to save pair {counter}")

d:\Deep_Learning_Object_Detection\MLOPs\pneumonia-segmentation\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
try:
    config = ConfigurationManger()
    data_transformation_config = config.get_data_transformation_config()
    data_transformation = DataTransformation(data_transformation_config)
    data_transformation.transform()
except Exception as e:
    raise CustomException(e, sys)

[2026-04-17 14:12:42,211: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-04-17 14:12:42,216: INFO: common: yaml file: params.yaml loaded successfully]
[2026-04-17 14:12:42,219: INFO: common: created directory at: artifacts]
[2026-04-17 14:12:42,220: INFO: common: created directory at: artifacts/data_transformation]


Processing NIfTI → PNG: 100%|██████████| 20/20 [00:14<00:00,  1.36it/s]

[2026-04-17 14:12:56,950: INFO: 4285949259: Data transformation completed: 535 pairs saved]
